# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset defined by a [Croissant](https://mlcommons.org/croissant/) schema, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, and all entities (record sets, fields, columns, etc.) are **referenced by their `@id`**.

*Dataset DOI*: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields (all referenced by `@id`).

In [ ]:
# List all record sets in the dataset schema
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print(f"Record sets (@id): {record_sets}\n")
# For each record set, print its associated fields (@id and name)
for rs_id in record_sets:
    rs_obj = next(r for r in metadata.to_json()['recordSet'] if r['@id']==rs_id)
    print(f"Fields in record set {rs_id}:")
    for field in rs_obj.get('field', []):
        print(f"  - {field['@id']} (name: {field.get('name', field['@id'])})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Choose one record set's `@id` from the overview above (the main table is typically the first one).** All field columns will be included.

In [ ]:
# Pick first record set (protein data table) for demonstration
if not record_sets:
    raise ValueError("No record sets found in the dataset.")
main_record_set_id = record_sets[0]  # change if you want a different record set

# Load records from the main record set
main_records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(main_records)
print(f"Fields/columns in record set {main_record_set_id} (by @id):\n")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by key attributes.

**All field references use the field's `@id`.**

In [ ]:
# Inspect numeric fields (by field @id)
numeric_fields = []
# Find numeric columns (float or int)
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_fields.append(col)

print(f"Numeric fields in {main_record_set_id}: {numeric_fields}")

# Select a numeric field for EDA (choose one that is present)
if not numeric_fields:
    raise ValueError("No numeric fields found for EDA.")
numeric_field_id = numeric_fields[0]  # use first numeric field

# Example: filter to records with value > threshold
threshold = df[numeric_field_id].mean() if pd.notna(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}):\n")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by a categorical field, if available
cat_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
if cat_fields:
    group_field_id = cat_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nAverage {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Below, we plot the normalized values and, if possible, a histogram and a boxplot grouped by a categorical field, using field `@id`s.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the original numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group if possible
if cat_fields:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

# Scatter plot if another numeric field exists
if len(numeric_fields) > 1:
    other_numeric = numeric_fields[1]
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=numeric_field_id, y=other_numeric, data=filtered_df)
    plt.title(f"{numeric_field_id} vs {other_numeric}")
    plt.show()

## 6. Conclusion
From this notebook, we've demonstrated how to use the Croissant schema and `mlcroissant` to:

- Load a public dataset **referencing entities strictly by `@id`**
- Enumerate available record sets and their fields
- Load tabular data for analysis, filter and normalize numeric data using column `@id`
- Quickly visualize field distributions and relationships

Explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset further by reproducing these steps for additional record sets and columns of interest!

----
*Notebook generated for reproducible, schema-driven data exploration using `mlcroissant`.*